# Held but Not Available — reproduce the results and try your own numbers

This notebook runs in Google Colab or a local Jupyter session.

1. **Setup** downloads the code (Colab only).
2. **Reproduce** recomputes every table, figure and number in Section 6 and checks them against the manuscript.
3. **Try your own parameters** lets you change any value in Table 1. Each result is cross-checked by three independent computations.
4. **Sweep** shows how the optimal reserve responds to one parameter.
5. **Tests** re-check the formulas on 40 random parameter sets (optional).

## 1. Setup

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/sophiateams2026/Optimal-Protection-Levels-in-a-Cross-Border-Two-Warehouse-Inventory-System"   # <- replaced with the public repository URL

if os.path.exists(os.path.join("..", "hbna")):   # opened from the notebooks/ folder
    os.chdir("..")
if not os.path.exists("hbna"):                   # Colab: fetch the repository
    if not os.path.exists("repo"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
    os.chdir("repo")
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))
%matplotlib inline
import numpy as np
import hbna, explore
from hbna import model
print("Working directory:", os.getcwd())

## 2. Reproduce Section 6

Takes about 20 seconds. The verification report lists every number reported in the manuscript next to the value computed here.

In [ ]:
!python reproduce_all.py
from IPython.display import Markdown, Image, display
display(Markdown(open("output/verification_report.md", encoding="utf-8").read()))

In [ ]:
for f in ["fig1.png", "fig2.png", "fig3a.png", "fig3b.png", "fig4.png"]:
    display(Image(f"output/{f}", width=650))

## 3. Try your own parameters

Edit any value below and run the cell. The baseline of Table 1 is shown.

* `p`, `ce`, `cb` and `dh` are optional shortcuts. Leave them as `None` to build them from their accounting components (`p = lam*T0`, `ce = compliance + delta*v`, `cb = r*v + r*P + penalty`, `dh = (sD - sO) + r*dv`).
* `extra_R` adds reserves of your choice to the cost table.

The output gives the regime and R\* from Theorem 1, a brute-force R\*, a Table 2 style breakdown, and a cross-check of the expected costs by three routes: (A) Lemma 2 averaged over the onset, (B) the closed forms of Lemma 4, (C) a time-stepping simulation of the issuing rule in Definition 1.

In [ ]:
values = dict(
    Q=1000,          # replenishment quantity
    d=100,           # demand per week
    QD=400,          # stock allocated to the demand-side warehouse (0 < QD < Q)
    v=50, P=80,      # landed unit cost, selling price
    dv=0,            # carrying-amount gap v_D - v_O (0 = importer; 5 = exporter of Section 6.4)
    r=0.002,         # cost of capital per week
    sD=0.20, sO=0.08,          # storage rates per unit-week
    compliance=0.10, delta=0.001,   # c_e = compliance + delta * v
    penalty=1.50,    # c_b = r v + r P + penalty
    kappa=5,         # per-unit backlog cost
    lam=0.02,        # restriction rate per week (p = lam * T0)
    L=3,             # restriction duration in weeks
    p=None, ce=None, cb=None, dh=None,   # optional shortcuts
)
extra_R = [150]

prm = explore.make_params(**{k: v for k, v in values.items() if v is not None})
result = explore.analyse(prm, extra_R=extra_R, show=True)

## 4. Sweep one parameter

Any name from the cell above can be swept, for example `L`, `p`, `kappa`, `ce`, `dh`, `QD`.

In [ ]:
header, rows = explore.sweep(prm, "L", 0.5, 8, 31, show=True)

## 5. Run the test suite (optional)

Includes 40 random parameter sets on which the closed forms, Theorem 1 and convexity are re-checked.

In [ ]:
!python -m pytest -q